In [3]:
# conda activate chronocell

import os
import sys

os.chdir("/mnt/lareaulab/reliscu/projects/Chronocell/analyses/janssens_2025_preprint")

sys.path.append("/mnt/lareaulab/reliscu/programs/FGP_2024")
sys.path.append("/mnt/lareaulab/reliscu/projects/Chronocell/analyses/janssens_2025_preprint/code")
# sys.path.append("/mnt/lareaulab/reliscu/projects/Chronocell/analyses/simulations/code")

import Chronocell
from reconstruct_RNA_history import *
# from protein_from_RNA import *

In [4]:
# Get traj object from running Chronocell
import pickle
with open("eLNPs_var>1.2_traj_WS.pkl", "rb") as f:
    traj = pickle.load(f)

In [5]:
Y = traj.X
Q = traj.Q[:, 0, :] 
tau = traj.tau # State transition times (global)
t = traj.t
theta = traj.theta
topo = traj.topo

theta_ = theta.copy()
a0 = theta_[:, 0] # Starting RNA abundance 
a = theta_[:, 1:len(topo.flatten())] 
beta = theta_[:, -2] # Splicing rate
alpha = a * beta[:, None] # These values are divided by splicing rate; removing this factor now
gamma = theta_[:, -1] # Degradation rate
state_grid = np.searchsorted(tau, t, side="left") - 1
state_grid[0] = 0 


In [6]:
# tau
# np.searchsorted(t, tau, side="left")

In [7]:
# U_max = 15
# S_max = 20

# states, index_for = enumerate_states(U_max, S_max)

# # Prep rate matrices

# A_per_gene = [] 
# for j in range(0, alpha.shape[0]):
#     A_for_this_gene = []
#     for i in range(0, alpha.shape[1]):
#         rxns = define_reactions(alpha[j, i], beta[j], gamma[j])
#         A = create_transition_matrix_sparse(rxns, states, index_for, U_max, S_max)
#         A_for_this_gene.append(A)
        
#     A_per_gene.append(A_for_this_gene)

In [8]:
# # Initialize X_fwd with stationary distribution (steady state at t=0)

# pi_per_gene = []

# for j in range(0, alpha.shape[0]):
#     alpha0 = a0[j] * beta[j]
#     rxns0 = define_reactions(alpha0, beta[j], gamma[j])
#     A0 = create_transition_matrix_sparse(rxns0, states, index_for, U_max, S_max)
#     pi = stationary_from_transition_matrix(A0)
#     pi_per_gene.append(pi)

In [9]:
# # Calc forward state probabilities for each gene

# X_fwd_per_gene = []

# for j in range(0, alpha.shape[0]):
#     X_fwd = forward_distribution(A_per_gene[j], pi_per_gene[j], states, t, tau, state_grid)
#     X_fwd_per_gene.append(X_fwd)

In [ ]:
gene_idx = 0

########

U_max = (np.max(Y[:, gene_idx, 0]) + 3).astype("int")
S_max = (np.max(Y[:, gene_idx, 1]) + 3).astype("int")
states, index_for = enumerate_states(U_max, S_max)

beta_j = beta[gene_idx]
gamma_j = gamma[gene_idx] 
alpha_j = alpha[gene_idx, :]

# Make generator matrix (per transcription rate)
A = []
for i in range(0, alpha.shape[1]):
    rxns = define_reactions(alpha_j[i], beta_j, gamma_j)
    A1 = create_transition_matrix(rxns, states, index_for, U_max, S_max)
    A.append(A1)

# Calculate forward probability distribution (needed for reverse generator)
alpha0 = a0[gene_idx] * beta_j
pi = stationary_from_params(alpha0, beta_j, gamma_j, states) 
X_fwd = forward_distribution(A, pi, states, t, tau, state_grid)

n_cells = Y.shape[0]

U_curr, S_curr = Y[:, gene_idx, 0], Y[:, gene_idx, 1]
X_curr = np.zeros(shape=(len(states), n_cells), dtype="float")
for cell_idx in range(n_cells):
    X_curr[index_for[(U_curr[cell_idx], S_curr[cell_idx])], cell_idx] = 1.0 
        
X_bw = np.zeros(shape=(len(states), len(t), Y.shape[0])) 

# Start backwards trajectory at cell's inferred position in time
t_obs = np.argmax(Q, axis=1)

for k in reversed(range(1, len(t))):
    t_prev, t_curr = t[k-1], t[k]
    state_prev, state_curr = state_grid[k-1], state_grid[k]
    
    mask = t_obs >= k
    X_bw[:, k, mask] = X_curr[:, mask]

    if state_prev == state_curr:
        dt = t_curr - t_prev
        M_rev = get_expm_rev_per_gene(state_curr, k, dt)
        X_prev = (X_curr[:, mask].T @ M_rev).T
        
    else:
        # State switch happens in current interval             
        t_s = tau[state_curr]

        # Split backward march into 2 steps
        dt2 = t_curr - t_s # right interval: (state_switch_time, t_k]
        M2_rev = get_expm_rev_per_gene(state_curr, k, dt2)
        X_mid = (X_curr[:, mask].T @ M2_rev).T
        
        dt1 = t_s - t_prev # left interval: (t_{k-1}, state_switch_time]
        M1_rev = get_expm_rev_per_gene(state_prev, k, dt1)
        X_prev = (X_mid @ M1_rev).T

    X_curr[:, mask] = X_prev
    X_bw[:, k-1, mask] = X_prev

In [ ]:
k = 99


In [17]:
np.argmax(Q, axis=1).shape

(21325,)

### Dense implementation

In [ ]:
X_bw_per_gene = []
states_per_gene = []
max = 300
    
for gene_idx in range(0, Y.shape[0]):
    print("Starting gene", gene_idx)

    # Set max # of RNAs based on observed values for a given gene
    U_max = np.max(Y[:, gene_idx, 0]).astype("int") + 1
    S_max = np.max(Y[:, gene_idx, 1]).astype("int") + 1
    states, index_for = enumerate_states(U_max, S_max)

    # if (U_max < max) & (S_max < max):
    
    beta_j = beta[gene_idx]
    gamma_j = gamma[gene_idx] 
    alpha_j = alpha[gene_idx, :]
    
    # Make generator matrix (per transcription rate)
    A = []
    for i in range(0, alpha.shape[1]):
        rxns = define_reactions(alpha_j[i], beta_j, gamma_j)
        A1 = create_transition_matrix(rxns, states, index_for, U_max, S_max)
        A.append(A1)
    
    # Calculate forward probability distribution (needed for reverse generator)
    alpha0 = a0[gene_idx] * beta_j
    pi = stationary_from_params(alpha0, beta_j, gamma_j, states) 
    X_fwd = forward_distribution(A, pi, states, t, tau, state_grid)

    # Calculate backward probability distribution (per cell)
    X_bw_per_cell = [] 
    for cell_idx in range(0, Q.shape[0]):
        X_bw = backward_distribution(Y, Q, gene_idx, cell_idx, states, index_for, t, tau, state_grid)
        X_bw_per_cell.append(X_bw)
        
    X_bw_per_gene.append(X_bw_per_cell)
    states_per_gene.append(states)
    
    get_expm_per_gene.cache_clear()
    get_A_rev.cache_clear()
    get_expm_rev_per_gene.cache_clear()
    

### Sparse implementation

In [ ]:
gene_idx = 330

U_max = (np.max(Y[:, gene_idx, 0]) + 3).astype("int")
S_max = (np.max(Y[:, gene_idx, 1]) + 3).astype("int")
states, index_for = enumerate_states(U_max, S_max)

beta_j = beta[gene_idx]
gamma_j = gamma[gene_idx] 
alpha_j = alpha[gene_idx, :]

# Make generator matrix (per transcription rate)
A = []
for i in range(0, alpha.shape[1]):
    rxns = define_reactions(alpha_j[i], beta_j, gamma_j)
    A1 = create_transition_matrix_sparse(rxns, states, index_for, U_max, S_max)
    A.append(A1)
    
# Calculate forward probability distribution (needed for reverse generator)
alpha0 = a0[gene_idx] * beta_j
pi = stationary_from_params(alpha0, beta_j, gamma_j, states)
 
import time

start = time.perf_counter()

X_fwd = forward_distribution_sparse(A, pi, states, t, tau, state_grid)

end = time.perf_counter()
print(f"Elapsed: {end - start:.4f} seconds")

In [ ]:
X_bw_per_gene = []
states_per_gene = []
    
for gene_idx in range(0, Y.shape[0]):
    print("Starting gene", gene_idx)

    # Set max # of RNAs based on observed values for a given gene
    U_max = np.max(Y[:, gene_idx, 0]).astype("int") + 1
    S_max = np.max(Y[:, gene_idx, 1]).astype("int") + 1
    states, index_for = enumerate_states(U_max, S_max)
    
    beta_j = beta[gene_idx]
    gamma_j = gamma[gene_idx] 
    alpha_j = alpha[gene_idx, :]
    
    # Make generator matrix (per transcription rate)
    A = []
    for i in range(0, alpha.shape[1]):
        rxns = define_reactions(alpha_j[i], beta_j, gamma_j)
        A1 = create_transition_matrix_sparse(rxns, states, index_for, U_max, S_max)
        A.append(A1)
     
    # Calculate forward probability distribution (needed for reverse generator)
    alpha0 = a0[gene_idx] * beta_j
    pi = stationary_from_params(alpha0, beta_j, gamma_j, states)
    X_fwd = forward_distribution_sparse(A, pi, states, t, tau, state_grid)
    
    # Calculate backward probability distribution (per cell)
    X_bw_per_cell = [] 
    for cell_idx in range(0, Q.shape[0]):
        X_bw = backward_distribution_sparse(Y, Q, gene_idx, cell_idx, states, index_for, t, tau, state_grid)
        X_bw_per_cell.append(X_bw)
        
    X_bw_per_gene.append(X_bw_per_cell)
    states_per_gene.append(states)
    
    get_A_rev_sparse.cache_clear()

In [ ]:
# Y_observed, Y, theta, rd, true_t, true_l = simulate_RNA(topo, tau, theta[0, :][None, :], n=20000, random_seed=666)